In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
from ase.build import bulk

In [9]:
from amstools.lammps import run_lammps_simulation, write_in_lammps

# Create structure with Al-Li

In [4]:
structure = bulk("Al", "fcc", cubic=True)*(2,2,2)

In [5]:
structure[0].symbol="Li"

In [6]:
structure[2].symbol="Li"

In [7]:
structure

Atoms(symbols='LiAlLiAl29', pbc=True, cell=[8.1, 8.1, 8.1])

# write_in_lammps
```
Docstring:
Write input scripts for a LAMMPS simulation as a sequence of steps  for the provided structure and ACE potential.

Note, that `working_dir` should be unique for given structure+list of steps.
Use rerun=True, if you want to rerun the simulation in the `working_dir`

Args:
    structure (ase.Atoms): The atomic structure for the simulation.
    potname (str): The name of the ACE interatomic potential file.
    working_dir (str): The working directory for the simulation.
        If the directory already contains output files from the specified steps
        and rerun is set to False, the simulation will not be re-run and the
        existing output files will be used.
    T (float, optional): The temperature (K) for the simulation. Defaults to 300.
    with_extrapolation_grade (bool, optional): Whether to compute ACE extrapolation grade. Defaults to False.
    dump_extrapolative (bool, optional): whether to dump extrapolative structures. Defaults to False.
    is_gpu (bool, optional): Whether to use GPU acceleration (pace/kk style). Defaults to False.
    lmp_exec (str or list, optional): The LAMMPS executable or a list containing the
        executable path and additional arguments. Defaults to "lmp".
    steps (list of dict, optional): A list of dictionaries specifying the simulation steps.
        Each dictionary should have the following keys:
            * type (str): The type of simulation step,
                e.g. "min_full", "min_atomic", "min_iso", "min_aniso", "eq", "mdmc".
            * name (str, optional): The name of the step for output identification.
                Defaults to the step type.
            * thermo_freq (int, optional): The frequency (in steps) for writing thermodynamic
                data during the step. Defaults to the value specified by the corresponding
                `n_*_thermo_freq` argument (e.g. `minimize_thermo_freq` for minimization steps).
            * Other keys (optional): Additional arguments specific to the step type
                (e.g. `T_init`, `T1`, `T2`, `P1`, `P2` for equilibration).
    n_min_steps (int, optional): The number of steps for minimization step. Defaults to 2500.
    n_eq_steps (int, optional): The number of steps for equilibration step. Defaults to 10000.
    n_mdmc_steps (int, optional): The number of steps for the molecular dynamics+Monte Carlo  (MDMC) stage.
              Defaults to 50000.
    minimize_thermo_freq (int, optional): The frequency (in steps) for writing thermodynamic
        data during minimization. Defaults to 50.
    equilibration_thermo_freq (int, optional): The frequency (in steps) for writing thermodynamic
        data during equilibration. Defaults to 250.
    mdmc_thermo_freq (int, optional): The frequency (in steps) for writing thermodynamic
        data during the MDMC. Defaults to 250.
    is_triclinic (bool, optional): Whether the simulation box is triclinic. Defaults to False.
            If structure is provided as ASE atoms, then this flag will be computed based on the structure itself.
    aniso_style (str, optional): The style for applying anisotropic pressure (e.g. "iso", "aniso", "tri").
        Defaults to None.
    specorder (list, optional): The species order for the simulation. If not provided, it will
        be inferred from the `structure` using `specorder_from_structure`.
    rerun (bool, optional): Whether to re-run the simulation even if the output files already
        exist in the working directory. Defaults to False.

Returns:
    tuple: A tuple containing the following elements:
        * energy (float): The total energy data from the last simulation step.
        * structure (ase.Atoms): The atomic structure after the last simulation step.


Examples:
    Here are examples of the `steps` parameter for different simulation protocols.
    You can copy and paste these into your Python code.

    .. code-block:: python

        # Example: Full relaxation (atomic positions and cell)
        steps = [
            {
                "type": "min_full",
                # "name": "my_full_relaxation",  # Optional: custom name for the step
                "thermo_freq": 100,           # Optional: thermodynamic data output frequency
                "steps": 1000,                # Optional: number of minimization steps
                "aniso_style": "aniso",       # Optional: 'iso', 'aniso', or 'tri'
            },
        ]

        # Example: Atomic relaxation only
        steps = [{"type": "min_atomic", "steps": 500}]

        # Example: Isotropic cell relaxation only
        steps = [{"type": "min_iso", "steps": 500}]

        # Example: Anisotropic cell relaxation only
        steps = [{"type": "min_aniso", "steps": 500}]

        # Example: NPT equilibration
        steps = [
            {
                "type": "eq",
                # "name": "my_npt_equilibration",
                "eq_type": "npt",             # 'npt', 'nvt', or 'nve'
                "thermo_freq": 250,
                "steps": 10000,
                "T_init": 600,                # Initial temperature for velocity creation
                "T1": 300, "T2": 300,          # Target temperature range
                "P1": 0, "P2": 0,              # Target pressure range (for NPT)
                "aniso_style": "iso",         # Pressure coupling style (for NPT)
                "seed": 12345,                # Random seed for velocity creation
                "dump": True,                 # Whether to dump trajectory
            },
        ]

        # Example: MDMC for atom swapping (e.g., for high-entropy alloys)
        steps = [
            {
                "type": "mdmc",
                # "name": "my_mdmc_swap",
                "thermo_freq": 500,
                "steps": 50000,
                "T_MDMC": 1000,               # Temperature for Monte Carlo swap attempts
                "T1": 1000, "T2": 1000,        # Temperature for MD (NPT)
                "P1": 0, "P2": 0,              # Pressure for MD (NPT)
                "aniso_style": "iso",
                # "elements": ["Ni", "Co", "Cr"], # Elements to be swapped, by default - all possible pairs
                "N": 100,                     # Perform M swap attempts every N steps
                "M": 100,                     # Number of swap attempts to perform
            },
        ]

        # Example: Tensile deformation along 'z' with NPT on other axes
        steps = [
            {
                "type": "deform_npt",
                # "name": "my_tensile_test",
                "thermo_freq": 100,
                "steps": 20000,
                "direction": "z",             # 'x', 'y', or 'z'
                "erate": 0.001,               # Strain rate
                "T1": 300, "T2": 300,          # Temperature for NPT
            },
        ]

        # Example: Tensile deformation along 'z' with NVT on other axes
        steps = [
            {
                    "type": "deform_nvt",
                    # "name": "my_tensile_test",
                    "thermo_freq": 100,
                    "steps": 20000,
                    "direction": "z",             # 'x', 'y', or 'z'
                    "erate": 0.001,               # Strain rate
                    "T1": 300, "T2": 300,          # Temperature for NPT
            },
        ]

        # Example: Shear deformation in 'xz' plane with NVT
        steps = [
            {
                "type": "shear_nvt",
                # "name": "my_shear_test",
                "thermo_freq": 100,
                "steps": 20000,
                "direction": "xz",            # 'xy', 'xz', or 'yz'
                "erate": 0.001,               # Strain rate
                "T1": 300, "T2": 300,          # Temperature for NVT
            },
        ]

        # Example: Grand Canonical Monte Carlo (GCMC)
        steps = [
            {
                "type": "gcmc",
                # "name": "my_gcmc_run",
                "thermo_freq": 250,
                "steps": 50000,
                "ensemble": "npt",            # 'npt' or 'nvt'
                "T": 500,                     # Temperature for ensemble and GCMC
                "P": 1.0,                     # Pressure for NPT ensemble
                "aniso_style": "iso",         # For NPT ('iso', 'aniso', 'tri')
                "elem": ["H", "He"],          # Element(s) to insert/delete. Can be a string or a list.
                "mu": [-2.5, -3.0],           # Chemical potential(s) of the reservoir(s). Must be a list if multiple elems.
                "N": 100,                     # Attempt GCMC every N steps. Can be a list.
                "X": 100,                     # Number of exchange attempts. Can be a list.
                "M": 0,                       # Number of move attempts (for inserted atoms). Can be a list.
                "displace": 0.0,              # Max MC translation distance. Can be a list.
                "overlap_cutoff": 1.0,        # Rejection distance for insertion. Can be a list.
                "seed": 12345,
            },
        ]

        # Example: Combined GCMC and Atom Swapping (MDMC)
        steps = [
            {
                "type": "gcmc_mdmc",
                # "name": "my_combined_run",
                "ensemble": "npt", "T": 800, "P": 1.0,
                # GCMC parameters
                "gcmc_elements": ["H"], "mu": [-1.5], "N": 100, "X": 100,
                # Atom Swap parameters
                "atom_swap_elements": ["Ni", "Co"],
                "swap_N": 100, "swap_M": 100,
                "swap_T": 800, # Can be different from MD temperature
            },
        ]
```

In [12]:
write_in_lammps(structure=structure, potname='/Users/lysogy36/.cache/grace/GRACE-2L-OMAT/',
                working_dir='lammps_calc',
                pairstyle='grace',
                steps=[
                        {
                            "type": "min_full",
                            # "name": "my_full_relaxation",  # Optional: custom name for the step
                            "thermo_freq": 100,           # Optional: thermodynamic data output frequency
                            "steps": 1000,                # Optional: number of minimization steps
                            "aniso_style": "aniso",       # Optional: 'iso', 'aniso', or 'tri'
                        },
    
                        {
                            "type": "eq",
                            # "name": "my_npt_equilibration",
                            "eq_type": "npt",             # 'npt', 'nvt', or 'nve'
                            "thermo_freq": 250,
                            "steps": 10000,
                            "T_init": 600,                # Initial temperature for velocity creation
                            "T1": 300, "T2": 300,          # Target temperature range
                            "P1": 0, "P2": 0,              # Target pressure range (for NPT)
                            "aniso_style": "iso",         # Pressure coupling style (for NPT)
                            "seed": 12345,                # Random seed for velocity creation
                            "dump": True,                 # Whether to dump trajectory
                        },
                    
                        {
                             "type": "mdmc",
                            # "name": "my_mdmc_swap",
                            "thermo_freq": 500,
                            "steps": 50000,
                            "T_MDMC": 1000,               # Temperature for Monte Carlo swap attempts
                            "T1": 1000, "T2": 1000,        # Temperature for MD (NPT)
                            "P1": 0, "P2": 0,              # Pressure for MD (NPT)
                            "aniso_style": "iso",
                            # "elements": ["Ni", "Co", "Cr"], # Elements to be swapped, by default - all possible pairs
                            "N": 100,                     # Perform M swap attempts every N steps
                            "M": 100,                     # Number of swap attempts to perform
                        },

                        {
                            "type": "deform_nvt",
                            # "name": "my_tensile_test",
                            "thermo_freq": 100,
                            "steps": 20000,
                            "direction": "z",             # 'x', 'y', or 'z'
                            "erate": 0.001,               # Strain rate
                            "T1": 300, "T2": 300,          # Temperature for NVT
                        },
                    
                ]
               )

Structure file (`structure.lammps-data`) and input (`in.lammps`) were written into working_dir

In [14]:
! ls lammps_calc/

in.lammps             structure.lammps-data


In [13]:
! cat lammps_calc/in.lammps


#-------------- INIT ---------------------
units		metal
dimension	3
boundary 	p p p
atom_style	atomic
variable 	dt equal 0.001

#---------------- ATOM DEFINITION -------------------
read_data	structure.lammps-data

mass 1 26.9815385 # Al
mass 2 6.94 # Li

pair_style grace

pair_coeff      * * /Users/lysogy36/.cache/grace/GRACE-2L-OMAT/ Al Li

neighbor	2.0 bin
neigh_modify	delay 0 every 1 check yes

# -------------------- min_dist --------------------
compute dist all pair/local dist
compute  min_dist all reduce  min c_dist inputs local

print "-------------- MINIMIZE  min_full ---------------------"

reset_timestep	0
thermo_style custom step cpuremain pe  fmax c_min_dist   press vol density pxx pyy pzz pxy pxz pyz xy xz yz
thermo	100
thermo_modify flush yes
dump	dump_relax all custom 100 dump.min_full.dump id type element xu yu zu
dump_modify dump_relax element Al Li

fix box_relax all box/relax aniso 0.0 vmax 0.001

min_style cg
minimize 0 1.0e-3 1000 1000

undump dump_relax

unfix b

Now, you can either transfer file or run it manually locally

# run_lammps_simulation

 Runs a LAMMPS simulation as a sequence of steps (within one LAMMPS script) for the provided structure and ACE potential

    Args:
        structure (ase.Atoms): The atomic structure for the simulation.
        potname (str): The name of the ACE interatomic potential file.
        working_dir (str): The working directory for the simulation.
            If the directory already contains output files from the specified steps
            and rerun is set to False, the simulation will not be re-run and the
            existing output files will be used.
        T (float, optional): The temperature (K) for the simulation. Defaults to 300.
        with_extrapolation_grade (bool, optional): Whether to compute ACE extrapolation grade. Defaults to False.
        is_gpu (bool, optional): Whether to use GPU acceleration (pace/kk style). Defaults to False.
        lmp_exec (str or list, optional): The LAMMPS executable or a list containing the
            executable path and additional arguments. Defaults to "lmp".
        steps (list of dict, optional): A list of dictionaries specifying the simulation steps.
            Each dictionary should have the following keys:
                * type (str): The type of simulation step,
                    e.g. "min_full", "min_atomic", "min_iso", "min_aniso", "eq", "mdmc".
                * name (str, optional): The name of the step for output identification.
                    Defaults to the step type.
                * thermo_freq (int, optional): The frequency (in steps) for writing thermodynamic
                    data during the step. Defaults to the value specified by the corresponding
                    `n_*_thermo_freq` argument (e.g. `minimize_thermo_freq` for minimization steps).
                * Other keys (optional): Additional arguments specific to the step type
                    (e.g. `T_init`, `T1`, `T2`, `P1`, `P2` for equilibration).
        n_min_steps (int, optional): The number of steps for minimization step. Defaults to 2500.
        n_eq_steps (int, optional): The number of steps for equilibration step. Defaults to 10000.
        n_mdmc_steps (int, optional): The number of steps for the molecular dynamics+Monte Carlo  (MDMC) stage.
                  Defaults to 50000.
        minimize_thermo_freq (int, optional): The frequency (in steps) for writing thermodynamic
            data during minimization. Defaults to 50.
        equlibration_thermo_freq (int, optional): The frequency (in steps) for writing thermodynamic
            data during equilibration. Defaults to 250.
        mdmc_thermo_freq (int, optional): The frequency (in steps) for writing thermodynamic
            data during the MDMC. Defaults to 250.
        is_triclinic (bool, optional): Whether the simulation box is triclinic. Defaults to False.
                If structure is provided as ASE atoms, then this flag will be computed based on the structure itself.
        aniso_style (str, optional): The style for applying anisotropic pressure (e.g. "iso", "aniso", "tri").
            Defaults to None.
        specorder (list, optional): The species order for the simulation. If not provided, it will
            be inferred from the `structure` using `specorder_from_structure`.
        rerun (bool, optional): Whether to re-run the simulation even if the output files already
            exist in the working directory. Defaults to False.

    Returns:
        tuple: A tuple containing the following elements:
            * energy (float): The total energy data from the last simulation step.
            * structure (ase.Atoms): The atomic structure after the last simulation step.


    Examples of steps:

    {"type": "min_*", #  "min_full", "min_atomic", "min_iso", "min_aniso"
         thermo_freq: minimize_thermo_freq, steps: n_min_steps, aniso_style: aniso_style }

    {"type": "eq",
        thermo_freq: minimize_thermo_freq, steps: n_eq_steps, aniso_style: aniso_style,
        T_init:2*T, T1: T, T2: T, P1: 0, P2: 0 }

    {"type": "mdmc",
        thermo_freq: minimize_thermo_freq, steps: n_min_steps, aniso_style: aniso_style
        T_MDMC: T,  T1: T, T2: T, P1: 0, P2: 0
       }

In [7]:
potname="/path/to/ACE/potential.yaml"

In [ ]:
lmp="/path/to/lmp"

NOTE. if you want to use MPI, then 
`lmp =["mpirun","-np","4","/path/to/MPI/lmp"]`

In [11]:
e_opt, structure_opt = run_lammps_simulation(
    structure=structure,
    working_dir="AlLi-example-1",
    potname=potname,
    lmp_exec=lmp,
    steps=["eq","mdmc","min_full"], 
    T=150,
    n_min_steps=250,
    n_eq_steps=500,
    n_mdmc_steps=500,
)

RUN `/home/users/lysogy36/CLionProjects/lammps-ace/build-A100/lmp -in in.lammps` in /home/users/lysogy36/PycharmProjects/ams-tools/tutorial/AlLi-example-1
Finish run


In [17]:
# Energy of final structure (eV)
e_opt

-108.4526855074338

In [18]:
# structure from final step
structure_opt

Atoms(symbols='Al11LiAl11LiAl8', pbc=True, cell=[8.074969387175793, 8.07499760058197, 8.075002898071315], id=..., masses=..., momenta=..., travel=..., type=...)

In [19]:
# compare to original structure
structure

Atoms(symbols='LiAlLiAl29', pbc=True, cell=[8.1, 8.1, 8.1])

In [ ]:
# Trying to rerun calculation on the same folder will read previous results:

In [27]:
e_opt_re, structure_opt_re = run_lammps_simulation(
    structure=structure,
    working_dir="AlLi-example-1",
    potname=potname,
    lmp_exec=lmp,
    steps=["eq","mdmc","min_full"],  # short notations can be used
    T=150,
    n_min_steps=250,
    n_eq_steps=500,
    n_mdmc_steps=500,
)

In [28]:
e_opt_re

-108.4526855074338

In [29]:
np.allclose(e_opt_re,e_opt)

True

In [30]:
structure_opt_re

Atoms(symbols='Al11LiAl11LiAl8', pbc=True, cell=[8.074969387175793, 8.07499760058197, 8.075002898071315], id=..., masses=..., momenta=..., travel=..., type=...)

In [31]:
structure_opt

Atoms(symbols='Al11LiAl11LiAl8', pbc=True, cell=[8.074969387175793, 8.07499760058197, 8.075002898071315], id=..., masses=..., momenta=..., travel=..., type=...)

To enforce rerun (if structure or samples have changed), use `rerun=True`

In [36]:
e_opt_rerun, structure_opt_rerun = run_lammps_simulation(
    structure=structure,
    working_dir="AlLi-example-1",
    potname=potname,
    lmp_exec=lmp,
    steps=["eq","mdmc","min_full"],  # short notations can be used
    T=150,
    n_min_steps=250,
    n_eq_steps=500,
    n_mdmc_steps=500,
    rerun=True
)

RUN `/home/users/lysogy36/CLionProjects/lammps-ace/build-A100/lmp -in in.lammps` in /home/users/lysogy36/PycharmProjects/ams-tools/tutorial/AlLi-example-1
Finish run


In [37]:
e_opt_rerun

-108.4526855074338

In [38]:
structure_opt_rerun

Atoms(symbols='Al11LiAl11LiAl8', pbc=True, cell=[8.074969387175793, 8.07499760058197, 8.075002898071315], id=..., masses=..., momenta=..., travel=..., type=...)

## Fine tuning, multiple steps

You can tune the steps, giving them different settings and names

In [39]:
e_opt_2, structure_opt_2 = run_lammps_simulation(
    structure=structure,
    working_dir="AlLi-example-2",
    potname=potname,
    lmp_exec=lmp,
    steps=[
        "eq", # step 1. Equilibration NPT at 300 K
        {"type": "eq", "name":"cooling",   "T1": 300, "T2": 1}, # step 2. Equilibration step, but cooling T=300K->1K
        "min_full" # step 3. full relaxation
    ], 
    T=300,
    n_min_steps=250,
    n_eq_steps=500,
    n_mdmc_steps=500,
)

In [40]:
e_opt_2

-108.3334640128103

In [41]:
structure_opt_2

Atoms(symbols='LiAlLiAl29', pbc=True, cell=[8.0785813245452, 8.075568605998326, 8.078570454478468], id=..., masses=..., momenta=..., travel=..., type=...)